In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/khakim/shakespeare/input.txt
/kaggle/input/datasets/khakim/pre-trained-character-level-gpt-3-6m-stats/loss_stats.pth
/kaggle/input/datasets/khakim/pre-trained-character-level-gpt-3-6m-stats/latest_checkpoint.pth
/kaggle/input/datasets/khakim/pre-trained-character-level-gpt-3-6m-stats/best_gpt_classic.pth
/kaggle/input/datasets/khakim/pre-trained-character-level-gpt-3-6m-stats/.virtual_documents/__notebook_source__.ipynb


In [37]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [38]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

Using device: cuda


In [39]:
class FeedForward(nn.Module):
    
    def __init__(self, embed_size, dropout = 0.1):
        super().__init__()
        self.lin1 = nn.Linear(embed_size, embed_size * 4)
        self.act = nn.SiLU()
        self.lin2 = nn.Linear(embed_size * 4, embed_size)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # (batch_size, block_size, embed_size)
        return self.dropout(self.lin2(self.act(self.lin1(x))))

class Block(nn.Module):
    
    def __init__(self, block_size, embed_size, num_head, dropout = 0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_size)
        self.attn = nn.MultiheadAttention(embed_size, num_head, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_size)
        self.ff = FeedForward(embed_size, dropout = dropout)
        self.dropout1 = nn.Dropout(dropout)
        mask = torch.triu(torch.ones(block_size, block_size), diagonal=1).bool()
        self.register_buffer('mask', mask)
    
    def forward(self, x):
        # (batch_size, block_size, embed_size)
        # adding residual connections!
        x = self.norm1(x)
        attMat = self.attn(key = x, query = x, value = x, attn_mask = self.mask[:x.shape[-2], :x.shape[-2]])[0]
        x = x + self.dropout1(attMat)
        x = x + self.ff(self.norm2(x))
        return x

class GPT(nn.Module):

    def __init__(self, block_size, alph_size, embed_size, num_head, num_blocks, dropout = 0.1):
        super().__init__()
        self.block_size = block_size
        self.embedding = nn.Embedding(alph_size, embed_size)
        self.pos_embedding = nn.Embedding(block_size, embed_size)
        self.dropout_embed = nn.Dropout(dropout)
        self.blocks = nn.Sequential(*[Block(block_size, embed_size, num_head, dropout) for _ in range(num_blocks)])
        self.norm = nn.LayerNorm(embed_size)
        self.linear = nn.Linear(embed_size, alph_size)
    
    def forward(self, x):
        # (batch_size, block_size), numbers from zero to alph_size - 1
        x_embed = self.embedding(x) # (batch_size, block_size, embed_size)
        x_pos_embed = self.pos_embedding(torch.arange(x.shape[-1], device = x.device)) # (block_size, embed_size)
        x_embed = self.dropout_embed(x_embed + x_pos_embed) # (batch_size, block_size, embed_size) + (block_size, embed_size) -> (batch_size, block_size, embed_size)
        logits = self.blocks(x_embed) # (batch_size, block_size, embed_size)
        logits = self.norm(logits) # (batch_size, block_size, embed_size)
        logits = self.linear(logits) # (batch_size, block_size, alph_size)
        return logits
        
    @torch.no_grad()
    def generate(self, context, gen_len):
        text = ''
        context = encode(context[-min(len(context), self.block_size):])
        for _ in range(gen_len):
            logits = self.__call__(torch.tensor([context], device = device))
            ix = torch.multinomial(F.softmax(logits[-1, -1], -1), num_samples=1).item()
            text += itos[ix]
            if len(context) == self.block_size:
                context = context[1:]
            context = context + [ix]
        return text

In [40]:
gen = torch.Generator().manual_seed(42)
embed_size = 192
num_head = 4
head_size = embed_size // num_head
num_blocks = 8
batch_size = 64
block_size = 384

In [41]:
text = open('/kaggle/input/datasets/khakim/shakespeare/input.txt').read()

alph =  sorted(list(set(text)))
alph_size = len(alph)
stoi = {c: i for i, c in enumerate(alph)}
itos = {v: k for k, v in stoi.items()}


def encode_ceaser(text, shift=1):
    result = []
    for c in text:
        if 'a' <= c <= 'z':
            result.append(chr((ord(c) - ord('a') + shift) % 26 + ord('a')))
        elif 'A' <= c <= 'Z':
            result.append(chr((ord(c) - ord('A') + shift) % 26 + ord('A')))
        else:
            result.append(c)
    return ''.join(result)

def decode_ceaser(text, shift=1):
    return encode_ceaser(text, shift=-shift)

encode = lambda x: [stoi[c] for c in x] 
decode = lambda x: ''.join([itos[ic] for ic in x])

n = int(0.9 * len(text))
Xtr = encode(encode_ceaser(text[:n]))
Xval = encode(encode_ceaser(text[n:]))

@torch.no_grad()
def getbatch(X):
    starts = torch.randint(0, len(X) - block_size, (batch_size,), generator = gen)
    context = torch.tensor([X[st : st + block_size] for st in starts]).to(device)
    target = torch.tensor([X[st + 1 : st + block_size + 1] for st in starts]).to(device)
    return context, target


In [42]:
best_model_data = torch.load('/kaggle/input/datasets/khakim/pre-trained-character-level-gpt-3-6m-stats/best_gpt_classic.pth', map_location = device)

In [43]:
best_model_data.keys()

dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'val_loss'])

In [44]:
model_state_dict = best_model_data['model_state_dict']
best_val_loss = best_model_data['val_loss']

In [45]:
ceaser_model = GPT(block_size, alph_size, embed_size, num_head, num_blocks).to(device)
ceaser_model.load_state_dict(model_state_dict)

<All keys matched successfully>

In [46]:
ceaser_model.eval()
print(ceaser_model.generate(' ', 500))

was o'clock hers;
One 'Has twere well alive.' I saw God pray with me.
If I had by spicious news from the Lord Angelo.

DUKE OF AUMERLE:
Doth not for my throne.

TYRREL:
Uncle, give this flatter pleasure; I come
To fight what is the repose.

CAPULERLET:
Ay fearful fear all be well? Come, come, lengtly.
Ay, but I see thy good sea
I friar'd: Yea, I will be much proceeding
And thy speech. But to speak ashes fair!
Come, come, friar Claudio; lost thou take't us,
For the rascnal of thou slewt thy side!


In [32]:
print(decode_ceaser(decode(x[0].tolist())))

ll the people:
in whose name myself
Attach thee as a traitorous innovator,
A foe to the public weal: obey, I charge thee,
And follow to thine answer.

CORIOLANUS:
Hence, old goat!

Senators, &C:
We'll surety him.

COMINIUS:
Aged sir, hands off.

CORIOLANUS:
Hence, rotten thing! or I shall shake thy bones
Out of thy garments.

SICINIUS:
Help, ye citizens!

MENENIUS:
On both sides mo


In [47]:
@torch.no_grad()
def getloss(X, num_samples):
    sumloss = 0
    with torch.autocast(device_type='cuda', dtype=torch.float16):
        for _ in range(num_samples):
            x, y = getbatch(X)
            logits = ceaser_model(x)
            loss = F.cross_entropy(logits.view(batch_size * block_size, alph_size), y.view(batch_size * block_size))
            sumloss += loss
    return sumloss / num_samples

In [48]:
params = list(ceaser_model.parameters())
lr = 3e-4
patience = 10  # сколько eval-шагов ждать без улучшения
patience_counter = 0
optimizer = torch.optim.AdamW(params, lr)
epochs = 6000
eval_every = 200
best_val_loss_ceaser = float('inf')
best_model_path = '/kaggle/working/best_gpt_ceaser.pth'
latest_checkpoint_path = '/kaggle/working/latest_checkpoint_ceaser.pth'

losses = []
train_losses = []
val_losses = []
scaler = torch.amp.GradScaler()
for e in range(epochs + 1):
    x, y = getbatch(Xtr)
    ceaser_model.train()
    optimizer.zero_grad()
    with torch.autocast(device_type='cuda', dtype=torch.float16):
        logits = ceaser_model(x)
        loss = F.cross_entropy(logits.view(batch_size * block_size, alph_size), y.view(batch_size * block_size))
        losses.append(loss.item())
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    if e % eval_every == 0:
        ceaser_model.eval()
        train_loss = getloss(Xtr, 50)
        val_loss = getloss(Xval, 50)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        print(f'Epoch {e}/{epochs}: train loss {train_loss}, val loss {val_loss}')
        torch.save({
                'epoch': e,
                'model_state_dict': ceaser_model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
            }, latest_checkpoint_path)
        if val_loss < best_val_loss_ceaser:
            best_val_loss_ceaser = val_loss
            patience_counter = 0
            torch.save({
                'epoch': e,
                'model_state_dict': ceaser_model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
            }, best_model_path)
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping at epoch {e}')
                break

torch.save({
            'train_losses': torch.tensor(train_losses),
            'val_losses': torch.tensor(val_losses),
            'all_losses': torch.tensor(losses)}, '/kaggle/working/loss_stats_ceaser.pth')

Epoch 0/6000: train loss 4.87842321395874, val loss 4.853082180023193
Epoch 200/6000: train loss 1.7129285335540771, val loss 1.8613382577896118
Epoch 400/6000: train loss 1.5180503129959106, val loss 1.7035367488861084
Epoch 600/6000: train loss 1.432118535041809, val loss 1.6323782205581665
Epoch 800/6000: train loss 1.3730778694152832, val loss 1.5817352533340454
Epoch 1000/6000: train loss 1.3313325643539429, val loss 1.5577937364578247
Epoch 1200/6000: train loss 1.2979530096054077, val loss 1.5460374355316162
Epoch 1400/6000: train loss 1.2724522352218628, val loss 1.522278904914856
Epoch 1600/6000: train loss 1.2461423873901367, val loss 1.5137840509414673
Epoch 1800/6000: train loss 1.228837251663208, val loss 1.5136680603027344
Epoch 2000/6000: train loss 1.2088017463684082, val loss 1.4998655319213867
Epoch 2200/6000: train loss 1.1891024112701416, val loss 1.4957358837127686
Epoch 2400/6000: train loss 1.1752450466156006, val loss 1.4965046644210815
Epoch 2600/6000: train lo

In [52]:
ceaser_model.eval()
res = ceaser_model.generate(' ', 1000)
print(res)

fu efbui.
P nz cpooz ebz bqpmmpboh cvu upbe!

IBTUJOHT:
Nz mpse, boe sje lopx J up ijn, J mppl'e cz uif dspxo
Up cfbs uif ifbsu, ofwfs eje mfbso.

Hbsefofs:
Obz, op, op, cvu mfu't bxfs, tjs.

IFOSZ CPMJOHCSPLF:
Uif ljoh't mjwf; gps uifz mpwf judift xfmm
Uibo dbnf uibu eje mpoh uifjs qbqfs.

EVLF PG ZPSL:
P, uifz uibu Mpoepo Dbnjmmp jo hsjfg,
Xifsf ibtu uibu, nbef uifz gbnpvt pg ijn.

EVLF PG ZPSL:
Bxbz, boe hfoumf tupm'o ipopvs tusjlf!
Uifz opu ep uifz ep tpgu tpvs tpmejfst.
Xfmdpnf njhiuz, gps uif djuz jt uifz bmm.

RVFFO FMJABCFUI:
Xfmm fokpz, cf uif cspuifs tbsmfu ojhiu.

LJOH SJDIBSE JJJ:
Opsgpml, ipx uibu jt tusffut nvtu txffu vocsjcf:
Mpset Sbhpaf opvsjoh upp Sbwfotqvshi,
Xifo Cpmjohcsplf nbz nz dpvousz gsjfoet,
Tuboe xjui uif npsojoh foet ijt fyfdvujpo,
Uijt jt npsf bmm tvsmz up efbui zpvs wpjdf,
Up ublf sbsf esbx pg nbssjbhft.

RVFFO FMJABCFUI:
Uif gsvjut pg ijt gpsft nbkftuz ifs opsuifst,
Qvu uif wbmbzfs't wpxt; boe cje if evuz,
Ps cz uifn uif efwpujpo pg uif xpsme,
Xijdi zpv 

In [53]:
print(decode_ceaser(res))

et death.
O my bonny day apolloang but toad!

HASTINGS:
My lord, and rid know I to him, I look'd by the crown
To bear the heart, never did learn.

Gardener:
Nay, no, no, but let's awer, sir.

HENRY BOLINGBROKE:
The king's live; for they love itches well
Than came that did long their paper.

DUKE OF YORK:
O, they that London Camillo in grief,
Where hast that, made they famous of him.

DUKE OF YORK:
Away, and gentle stol'n honour strike!
They not do they do soft sour soldiers.
Welcome mighty, for the city is they all.

QUEEN ELIZABETH:
Well enjoy, be the brother sarlet night.

KING RICHARD III:
Norfolk, how that is streets must sweet unbribe:
Lords Ragoze nouring too Ravenspurgh,
When Bolingbroke may my country friends,
Stand with the morning ends his execution,
This is more all surly to death your voice,
To take rare draw of marriages.

QUEEN ELIZABETH:
The fruits of his fores majesty her northers,
Put the valayer's vows; and bid he duty,
Or by them the devotion of the world,
Which you 